# 03 - Category Classification

Goal: train a separate ticket category classifier using TF-IDF features.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from preprocessing import add_cleaned_ticket_column
from train import build_vectorizer, get_candidate_models, RANDOM_STATE, TEST_SIZE
from utils import load_raw_data, prepare_ticket_dataframe

In [ ]:
df = add_cleaned_ticket_column(prepare_ticket_dataframe(load_raw_data()))
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_ticket'], df['category'], test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=df['category']
)
vectorizer = build_vectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
X_train.shape

In [ ]:
models = get_candidate_models(include_naive_bayes=True)
reports = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    reports[name] = classification_report(y_test, preds, zero_division=0)
    print('\n' + '=' * 80)
    print(name)
    print(reports[name])

In [ ]:
best_model_name = 'LinearSVC'
best_model = models[best_model_name]
preds = best_model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, preds, xticks_rotation=45, cmap='Blues')
plt.title(f'Category Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.show()

## Interpretation

LinearSVC often performs strongly on sparse TF-IDF text because it can learn effective linear boundaries across many unigram and bigram features. Use macro F1-score, not only accuracy, when classes are imbalanced.